In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load your API key from .env
load_dotenv()

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "What is the capital of the United States?"}]
)

print(response.choices[0].message.content)


The capital of the United States is Washington, D.C.


# Nielson 2015 conversion rates to oral morphine equivalent

In [ ]:
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Dict

# Function to load Nielson conversions from a JSON file
def load_nielson_conversions(json_path: str) -> Dict[str, float]:
    with open(json_path, "r") as file:
        data = json.load(file)
    # Use the keys from the JSON directly as the dictionary keys
    return {key: value["conversion_factor"] for key, value in data.items()}

# Load the Nielson conversions globally
NIELSON_CONVERSIONS = load_nielson_conversions(Path("nielson_conversions.json"))


In [ ]:
# x-axis: dose of morphine, y-axis: inferred conversion coefficient
# x-axis: dose or morphine, y-axis: ratio of recommended opioid to Nielson recommendation

# DATAFRAME:
    ## PROMPT includes NIELSON conversion coefficient
    ## PROMPT includes EXAMPLES
    ## ADMINISTRATION_METHOD
    ## SUBSTANCE
    ## DOSE_FROM (random based on 10 mg to 120 mg oral morphine)
    ## EXPECTED DOSE_TO
    ## OBSERVED DOSE_TO FROM LLM
    ## RATIO OBSERVED/EXPECTED

# AGGREGATES:
    ## PROMPT includes NIELSON conversion coefficient
    ## PROMPT includes EXAMPLES
    ## ADMINISTRATION_METHOD
    ## SUBSTANCE
    ## PROPORTION of RATIO outside of 0.7 - 1.3

    # p-value for a non-zero slope
    # p-value for a non-one mean

    # CDF of RATIO EXPECTED/OBSERVED by including Nielson, including examples

In [ ]:
# helper function
# generate a random dose of opioid based on OME
# choose a random value on a log scale between 10 mg and 120 mg
# use Neilson 2015 to convert from OME to DOSE_FROM

In [ ]:
# prompt function
# generate a prompt from inputs:
    # INCLUDE NIELSON?
    # INCLUDE 10 mg and 120 mg EXAMPLES?
    # ADMINISTRATION_METHOD

In [ ]:
# an OPIOID_FROM and an OPIOID_TO both have a ADMINISTRATION_METHOD and SUBSTANCE
# ask to convert from DOSE_FROM of OPIOID_FROM to OPIOID_TO
# include a conversion coefficient indicating the oral morphine equivalent

In [ ]:
# Create a prompted_conversion object equivalent to a row of the dataframe
# has properties:
                
    ## ADMINISTRATION_METHOD_FROM
    ## SUBSTANCE_FROM
        ## DOSE_FROM (random based on 10 mg to 120 mg oral morphine)

    ## ADMINISTRATION_METHOD_TO (default oral)
    ## SUBSTANCE_TO (default morphine)
        ## EXPECTED DOSE_TO ACCORDING TO NIELSON
            ## PROMPT includes NIELSON conversion coefficient
            ## PROMPT includes EXAMPLES
                # PROMPT
                # MODEL PROMPTED
                    # RESPONSE
                    # OBSERVED DOSE_TO
                    # RATIO OBSERVED/EXPECTED
                    # RATIO INSIDE 0.7 - 1.3

In [ ]:
# Create an experiment object
    # INPUT:
        # TUPLES of (FROM, TO) or a LIST of FROM, defaulting to morphine
        # NUMBER of REPETITIONS
        # INCLUDE NIELSON?
        # INCLUDE 10 mg and 120 mg EXAMPLES?
    # OUTPUT:
        # LIST of prompted_conversions
        # DATAFRAME of prompted_conversions
        # AGGREGATES
            # by (FROM, TO) and total
                # Regression of EXPECTED DOSE vs. RATIO OBSERVED/EXPECTED
                # p-value non-zero slope
                # p-value non-one mean
            # by (FROM, TO) and total
                # CDF of RATIO OBSERVED/EXPECTED
            # by (FROM, TO) and total
                # Proportion RATIO INSIDE 0.7 - 1.3            
        # FIGURES
            # Scatter plot of EXPECTED DOSE v. RATIO OBSERVED/EXPECTED by (FROM, TO)
                # Should we include a regression line?
            # CDF of RATIO OBSERVED/EXPECTED by (FROM, TO)
            # Proportion of RATIO INSIDE 0.7 - 1.3 by (FROM, TO)



In [ ]:
from typing import List, Tuple, Union
from dataclasses import dataclass
import numpy as np

@dataclass
class Opioid:
    administration_method: str
    substance: str

@dataclass
class PromptedOpioidConversion:
    from_opioid: Opioid
    to_opioid: Opioid = Opioid(administration_method="oral", substance="morphine")
    dose_from: float = None
    expected_dose_to: float = None
    include_nielson: bool = True
    include_examples: bool = True
    prompt: str = None
    observed_dose_to: float = None
    ratio_observed_expected: float = None

class ConversionService:
    def __init__(self, nielson_conversions: Dict[str, float]):
        self.nielson_conversions = nielson_conversions

    def generate_random_dose(self, from_opioid: Opioid, dose_range: Tuple[float, float] = (10, 120)) -> float:
        """
        Generate a random dose for the given opioid within the specified range.

        Args:
            from_opioid (Opioid): The opioid for which the dose is being generated.
            dose_range (Tuple[float, float]): The range (min, max) for the dose in oral morphine equivalents.

        Returns:
            float: The generated dose in oral morphine equivalents, adjusted for the opioid.
        """
        min_dose, max_dose = dose_range
        oral_morphine_dose = np.exp(np.random.uniform(np.log(min_dose), np.log(max_dose)))
        opioid_key = f"{from_opioid.administration_method}_{from_opioid.substance}"

        if opioid_key not in self.nielson_conversions:
            raise KeyError(f"Opioid key '{opioid_key}' not found in Nielson conversions.")
        
        conversion_factor = self.nielson_conversions[opioid_key]
        return oral_morphine_dose / conversion_factor
    
    def calculate_expected_dose(self, dose_from: float, from_opioid: Opioid, to_opioid: Opioid) -> float:
        """
        Calculate the expected dose of the target opioid based on the dose of the source opioid.

        Args:
            dose_from (float): The dose of the source opioid.
            from_opioid (Opioid): The source opioid.
            to_opioid (Opioid): The target opioid.

        Returns:
            float: The expected dose of the target opioid.
        """
        from_key = f"{from_opioid.administration_method}_{from_opioid.substance}"
        to_key = f"{to_opioid.administration_method}_{to_opioid.substance}"

        if from_key not in self.nielson_conversions or to_key not in self.nielson_conversions:
            raise KeyError(f"Opioid keys '{from_key}' or '{to_key}' not found in Nielson conversions.")
        
        from_conversion = self.nielson_conversions[from_key]
        to_conversion = self.nielson_conversions[to_key]

        # Convert dose_from to oral morphine equivalent, then to target opioid
        oral_morphine_equivalent = dose_from * from_conversion
        expected_dose_to = oral_morphine_equivalent / to_conversion
        return expected_dose_to